
# Notebook 05 — Supplier & Manufacturing Analysis

## Objective
Analyze the sourcing and manufacturing decisions
generated by the mathematical optimization model.

## Business Questions
1. How did supplier allocations change?
2. Which suppliers drive procurement costs?
3. Where are supplier capacity constraints tight?
4. How did optimized production schedules change?
5. What explains the cost savings versus baseline?
6. Which operational bottlenecks deserve attention?

## Methodology
Compare the rule-based baseline from Notebook 03
with the validated optimizer outputs from Notebook 04.

All results use hypothetical supply network data.

## Deliverables
- Supplier allocation comparison
- Supplier utilization analysis
- Manufacturing capacity analysis
- Cost improvement decomposition
- Executive recommendations
- Reusable CSV reporting datasets
```

That's our only introductory Markdown cell. We'll add another at the very end.

In [1]:
# STEP 2: Load baseline and optimized planning datasets

import pandas as pd
import numpy as np
from pathlib import Path

# Locate Project 2 repository
current_path = Path.cwd().resolve()

repo_root = next(
    (
        path for path in [current_path, *current_path.parents]
        if (path / "data/assumptions/bom.csv").is_file()
    ),
    None
)

assert repo_root is not None, "Project 2 repository not found"

# Inputs for supplier and manufacturing analysis
files = {
    "suppliers": "data/assumptions/suppliers.csv",
    "plants": "data/assumptions/plants.csv",
    "bom": "data/assumptions/bom.csv",

    "baseline_orders":
        "outputs/tables/baseline_purchase_orders.csv",

    "optimized_orders":
        "outputs/tables/optimized_purchase_orders.csv",

    "baseline_production":
        "outputs/tables/baseline_production_plan.csv",

    "optimized_production":
        "outputs/tables/optimized_production_plan.csv",

    "baseline_costs":
        "outputs/tables/baseline_weekly_costs.csv",

    "optimized_costs":
        "outputs/tables/optimized_cost_breakdown.csv",

    "baseline_shipments":
        "outputs/tables/baseline_shipments.csv",

    "optimized_shipments":
        "outputs/tables/optimized_shipments.csv"
}

# Load all files
data = {}

for name, relative_path in files.items():

    file_path = repo_root / relative_path

    assert file_path.is_file(), (
        f"Missing dataset: {file_path}"
    )

    data[name] = pd.read_csv(file_path)

# Validate data completeness
assert len(data) == 11

assert len(data["baseline_production"]) == 12
assert len(data["optimized_production"]) == 12

# Reconcile financial totals
baseline_cost = data["baseline_costs"][
    "Total_Cost_USD"
].sum()

optimized_cost = data["optimized_costs"][
    "Cost_USD"
].sum()

cost_savings = baseline_cost - optimized_cost

assert np.isclose(
    baseline_cost,
    414139.70,
    atol=0.01
)

assert np.isclose(
    optimized_cost,
    404131.20,
    atol=0.01
)

print("NOTEBOOK 05 INITIALIZED SUCCESSFULLY")
print("=" * 45)
print("Datasets loaded:", len(data))
print("Baseline cost: $", round(baseline_cost, 2))
print("Optimized cost: $", round(optimized_cost, 2))
print("Cost savings: $", round(cost_savings, 2))

display(pd.DataFrame([
    {"Dataset": name, "Records": len(df)}
    for name, df in data.items()
]))


NOTEBOOK 05 INITIALIZED SUCCESSFULLY
Datasets loaded: 11
Baseline cost: $ 414139.7
Optimized cost: $ 404131.2
Cost savings: $ 10008.5


,Dataset,Records
0,suppliers,6
1,plants,1
2,bom,3
3,baseline_orders,51
4,optimized_orders,72
5,baseline_production,12
6,optimized_production,12
7,baseline_costs,12
8,optimized_costs,4
9,baseline_shipments,22


In [2]:
# STEP 3A: Compare baseline and optimized supplier allocation

baseline_orders = data["baseline_orders"]
optimized_orders = data["optimized_orders"]

# Aggregate baseline purchasing decisions
baseline_allocation = (
    baseline_orders.groupby(
        ["Supplier_ID", "Component_ID"],
        as_index=False
    )
    .agg(
        Baseline_Units=("Order_Units", "sum"),
        Baseline_Cost_USD=("Purchase_Cost_USD", "sum")
    )
)

# Aggregate optimized purchasing decisions
optimized_allocation = (
    optimized_orders.groupby(
        ["Supplier_ID", "Component_ID"],
        as_index=False
    )
    .agg(
        Optimized_Units=("Order_Units", "sum"),
        Optimized_Cost_USD=("Purchase_Cost_USD", "sum")
    )
)

# Compare both plans
supplier_comparison = baseline_allocation.merge(
    optimized_allocation,
    on=["Supplier_ID", "Component_ID"],
    how="outer",
    validate="one_to_one"
)

# Fill missing combinations, if any
numeric_columns = [
    "Baseline_Units",
    "Baseline_Cost_USD",
    "Optimized_Units",
    "Optimized_Cost_USD"
]

supplier_comparison[numeric_columns] = (
    supplier_comparison[numeric_columns].fillna(0)
)

# Quantify changes
supplier_comparison["Unit_Change"] = (
    supplier_comparison["Optimized_Units"]
    - supplier_comparison["Baseline_Units"]
)

supplier_comparison["Cost_Savings_USD"] = (
    supplier_comparison["Baseline_Cost_USD"]
    - supplier_comparison["Optimized_Cost_USD"]
)

# Validate financial reconciliation
assert np.isclose(
    supplier_comparison["Baseline_Cost_USD"].sum(),
    baseline_orders["Purchase_Cost_USD"].sum()
)

assert np.isclose(
    supplier_comparison["Optimized_Cost_USD"].sum(),
    optimized_orders["Purchase_Cost_USD"].sum()
)

print("SUPPLIER ALLOCATION COMPARISON")
print("=" * 45)

display(supplier_comparison.round(2))

print("\nTOTAL PROCUREMENT SAVINGS: $",
      round(supplier_comparison["Cost_Savings_USD"].sum(), 2))

print("\nSTEP 3A: PASSED")


SUPPLIER ALLOCATION COMPARISON


,Supplier_ID,Component_ID,Baseline_Units,Baseline_Cost_USD,Optimized_Units,Optimized_Cost_USD,Unit_Change,Cost_Savings_USD
0,SUP_A,BATT_001,4050,56700,4084,57176.0,34,-476.0
1,SUP_A,ELEC_001,4050,129600,4084,130688.0,34,-1088.0
2,SUP_A,ENCL_001,4050,32400,4084,32672.0,34,-272.0
3,SUP_B,BATT_001,515,9270,481,8658.0,-34,612.0
4,SUP_B,ELEC_001,515,20600,481,19240.0,-34,1360.0
5,SUP_B,ENCL_001,515,5665,481,5291.0,-34,374.0



TOTAL PROCUREMENT SAVINGS: $ 510.0

STEP 3A: PASSED


In [3]:
# STEP 3B: Analyze supplier capacity utilization

supplier_master = data["suppliers"]

planning_weeks = 12

# Total available capacity over 12 weeks
supplier_capacity = (
    supplier_master.groupby("Supplier_ID")
    ["Weekly_Capacity_Units"]
    .sum()
    * planning_weeks
)

# Baseline purchased units
baseline_volume = (
    baseline_orders.groupby("Supplier_ID")
    ["Order_Units"].sum()
)

# Optimized purchased units
optimized_volume = (
    optimized_orders.groupby("Supplier_ID")
    ["Order_Units"].sum()
)

# Create supplier utilization comparison
supplier_utilization = pd.DataFrame({
    "Capacity_12_Weeks": supplier_capacity,
    "Baseline_Units": baseline_volume,
    "Optimized_Units": optimized_volume
}).fillna(0)

supplier_utilization["Baseline_Utilization_Pct"] = (
    supplier_utilization["Baseline_Units"]
    / supplier_utilization["Capacity_12_Weeks"]
    * 100
)

supplier_utilization["Optimized_Utilization_Pct"] = (
    supplier_utilization["Optimized_Units"]
    / supplier_utilization["Capacity_12_Weeks"]
    * 100
)

supplier_utilization["Utilization_Change_Pct_Points"] = (
    supplier_utilization["Optimized_Utilization_Pct"]
    - supplier_utilization["Baseline_Utilization_Pct"]
)

# Validate supplier utilization
assert supplier_utilization[
    "Optimized_Utilization_Pct"
].between(0, 100).all()

assert supplier_utilization[
    "Baseline_Utilization_Pct"
].between(0, 100).all()

print("SUPPLIER CAPACITY UTILIZATION")
print("=" * 45)

display(supplier_utilization.round(2))

print("\nSTEP 3B: PASSED")


SUPPLIER CAPACITY UTILIZATION


,Capacity_12_Weeks,Baseline_Units,Optimized_Units,Baseline_Utilization_Pct,Optimized_Utilization_Pct,Utilization_Change_Pct_Points
Supplier_ID,,,,,,
SUP_A,16200,12150,12252,75.00,75.63,0.63
SUP_B,9000,1545,1443,17.17,16.03,-1.13



STEP 3B: PASSED


In [4]:
# STEP 4A: Analyze weekly supplier capacity pressure

# Create every supplier-component-week combination
supplier_weeks = data["suppliers"][[
    "Supplier_ID",
    "Component_ID",
    "Weekly_Capacity_Units",
    "Lead_Time_Weeks"
]].merge(
    pd.DataFrame({"Week": range(1, 13)}),
    how="cross"
)

# Baseline orders by week
baseline_weekly = (
    data["baseline_orders"]
    .groupby(
        ["Supplier_ID", "Component_ID", "Week"],
        as_index=False
    )["Order_Units"]
    .sum()
    .rename(columns={
        "Order_Units": "Baseline_Units"
    })
)

# Optimized orders by week
optimized_weekly = (
    data["optimized_orders"]
    .groupby(
        ["Supplier_ID", "Component_ID", "Order_Week"],
        as_index=False
    )["Order_Units"]
    .sum()
    .rename(columns={
        "Order_Week": "Week",
        "Order_Units": "Optimized_Units"
    })
)

# Combine all purchasing decisions
supplier_weekly = supplier_weeks.merge(
    baseline_weekly,
    on=["Supplier_ID", "Component_ID", "Week"],
    how="left",
    validate="one_to_one"
).merge(
    optimized_weekly,
    on=["Supplier_ID", "Component_ID", "Week"],
    how="left",
    validate="one_to_one"
)

supplier_weekly[[
    "Baseline_Units",
    "Optimized_Units"
]] = supplier_weekly[[
    "Baseline_Units",
    "Optimized_Units"
]].fillna(0)

# Calculate utilization
supplier_weekly["Baseline_Utilization_Pct"] = (
    supplier_weekly["Baseline_Units"]
    / supplier_weekly["Weekly_Capacity_Units"]
    * 100
)

supplier_weekly["Optimized_Utilization_Pct"] = (
    supplier_weekly["Optimized_Units"]
    / supplier_weekly["Weekly_Capacity_Units"]
    * 100
)

# Identify fully utilized supplier capacity
supplier_weekly["At_Capacity"] = np.isclose(
    supplier_weekly["Optimized_Units"],
    supplier_weekly["Weekly_Capacity_Units"]
)

# Validate capacity compliance
assert supplier_weekly[
    "Optimized_Utilization_Pct"
].between(0, 100).all()

assert supplier_weekly[
    "Baseline_Utilization_Pct"
].between(0, 100).all()

assert len(supplier_weekly) == 72

print("WEEKLY SUPPLIER CAPACITY ANALYSIS")
print("=" * 45)

print("Records analyzed:", len(supplier_weekly))
print(
    "Optimized supplier-component weeks at capacity:",
    supplier_weekly["At_Capacity"].sum()
)

print("\nHIGHEST CAPACITY PRESSURE:")
display(
    supplier_weekly.sort_values(
        "Optimized_Utilization_Pct",
        ascending=False
    ).head(15)
)

print("\nSTEP 4A: PASSED")


WEEKLY SUPPLIER CAPACITY ANALYSIS
Records analyzed: 72
Optimized supplier-component weeks at capacity: 29

HIGHEST CAPACITY PRESSURE:


,Supplier_ID,Component_ID,Weekly_Capacity_Units,Lead_Time_Weeks,Week,Baseline_Units,Optimized_Units,Baseline_Utilization_Pct,Optimized_Utilization_Pct,At_Capacity
0,SUP_A,ELEC_001,450,2,1,450.0,450,100.0,100.0,True
16,SUP_A,BATT_001,450,2,5,450.0,450,100.0,100.0,True
38,SUP_B,ELEC_001,250,1,3,65.0,250,26.0,100.0,True
32,SUP_A,ENCL_001,450,2,9,450.0,450,100.0,100.0,True
31,SUP_A,ENCL_001,450,2,8,450.0,450,100.0,100.0,True
30,SUP_A,ENCL_001,450,2,7,450.0,450,100.0,100.0,True
29,SUP_A,ENCL_001,450,2,6,450.0,450,100.0,100.0,True
28,SUP_A,ENCL_001,450,2,5,450.0,450,100.0,100.0,True
27,SUP_A,ENCL_001,450,2,4,450.0,450,100.0,100.0,True
26,SUP_A,ENCL_001,450,2,3,450.0,450,100.0,100.0,True



STEP 4A: PASSED


In [5]:
# STEP 4B: Summarize supplier capacity pressure

supplier_pressure = (
    supplier_weekly.groupby(
        ["Supplier_ID", "Component_ID"],
        as_index=False
    )
    .agg(
        Total_Optimized_Orders=("Optimized_Units", "sum"),
        Peak_Utilization_Pct=(
            "Optimized_Utilization_Pct", "max"
        ),
        Weeks_At_Capacity=("At_Capacity", "sum"),
        Average_Utilization_Pct=(
            "Optimized_Utilization_Pct", "mean"
        )
    )
)

supplier_pressure = supplier_pressure.sort_values(
    ["Weeks_At_Capacity", "Peak_Utilization_Pct"],
    ascending=False
)

print("SUPPLIER CAPACITY PRESSURE SUMMARY")
display(supplier_pressure.round(2))

assert len(supplier_pressure) == 6

print("\nSTEP 4B: PASSED")


SUPPLIER CAPACITY PRESSURE SUMMARY


,Supplier_ID,Component_ID,Total_Optimized_Orders,Peak_Utilization_Pct,Weeks_At_Capacity,Average_Utilization_Pct
0,SUP_A,BATT_001,4084,100.0,9,75.63
1,SUP_A,ELEC_001,4084,100.0,9,75.63
2,SUP_A,ENCL_001,4084,100.0,9,75.63
4,SUP_B,ELEC_001,481,100.0,1,16.03
5,SUP_B,ENCL_001,481,100.0,1,16.03
3,SUP_B,BATT_001,481,90.4,0,16.03



STEP 4B: PASSED


In [6]:
# STEP 5A: Compare baseline and optimized manufacturing

baseline_production = data["baseline_production"]
optimized_production = data["optimized_production"]

# Build weekly comparison
manufacturing_comparison = baseline_production[[
    "Week", "Production_Units"
]].rename(columns={
    "Production_Units": "Baseline_Production"
}).merge(
    optimized_production[[
        "Week", "Production_Units"
    ]].rename(columns={
        "Production_Units": "Optimized_Production"
    }),
    on="Week",
    validate="one_to_one"
)

# Retrieve manufacturing capacity
plant_capacity = int(
    data["plants"]["Weekly_Capacity_Units"].iloc[0]
)

manufacturing_comparison["Plant_Capacity"] = plant_capacity

manufacturing_comparison["Production_Change"] = (
    manufacturing_comparison["Optimized_Production"]
    - manufacturing_comparison["Baseline_Production"]
)

manufacturing_comparison["Baseline_Utilization_Pct"] = (
    manufacturing_comparison["Baseline_Production"]
    / plant_capacity * 100
)

manufacturing_comparison["Optimized_Utilization_Pct"] = (
    manufacturing_comparison["Optimized_Production"]
    / plant_capacity * 100
)

manufacturing_comparison["Unused_Capacity"] = (
    plant_capacity
    - manufacturing_comparison["Optimized_Production"]
)

# Identify weeks operating at maximum capacity
manufacturing_comparison["At_Capacity"] = np.isclose(
    manufacturing_comparison["Optimized_Production"],
    plant_capacity
)

# Validate
assert len(manufacturing_comparison) == 12

assert manufacturing_comparison[
    "Optimized_Production"
].le(plant_capacity).all()

assert np.isclose(
    manufacturing_comparison["Baseline_Production"].sum(),
    manufacturing_comparison["Optimized_Production"].sum()
)

print("MANUFACTURING CAPACITY ANALYSIS")
print("=" * 45)

display(manufacturing_comparison.round(2))

print("\nMANUFACTURING SUMMARY:")
print(
    "Total optimized production:",
    manufacturing_comparison["Optimized_Production"].sum()
)
print(
    "Peak weekly production:",
    manufacturing_comparison["Optimized_Production"].max()
)
print(
    "Weeks at maximum capacity:",
    manufacturing_comparison["At_Capacity"].sum()
)
print(
    "Average utilization:",
    round(
        manufacturing_comparison[
            "Optimized_Utilization_Pct"
        ].mean(), 2
    ), "%"
)

print("\nSTEP 5A: PASSED")


MANUFACTURING CAPACITY ANALYSIS


,Week,Baseline_Production,Optimized_Production,Plant_Capacity,Production_Change,Baseline_Utilization_Pct,Optimized_Utilization_Pct,Unused_Capacity,At_Capacity
0,1,515,580,580,65,88.79,100.00,0,True
1,2,515,580,580,65,88.79,100.00,0,True
2,3,515,546,580,31,88.79,94.14,34,False
3,4,515,580,580,65,88.79,100.00,0,True
4,5,515,450,580,-65,88.79,77.59,130,False
5,6,515,570,580,55,88.79,98.28,10,False
6,7,515,525,580,10,88.79,90.52,55,False
7,8,515,450,580,-65,88.79,77.59,130,False
8,9,515,450,580,-65,88.79,77.59,130,False
9,10,515,450,580,-65,88.79,77.59,130,False



MANUFACTURING SUMMARY:
Total optimized production: 5665
Peak weekly production: 580
Weeks at maximum capacity: 3
Average utilization: 81.39 %

STEP 5A: PASSED


In [7]:
# STEP 5B: Analyze material inventory buffers

# Load component inventory ledgers
baseline_component = pd.read_csv(
    repo_root /
    "outputs/tables/baseline_component_inventory.csv"
)

optimized_component = pd.read_csv(
    repo_root /
    "outputs/tables/optimized_component_inventory.csv"
)

# Compare weekly component inventory
component_buffer_comparison = baseline_component[[
    "Week", "Component_ID", "Ending_Inventory"
]].rename(columns={
    "Ending_Inventory": "Baseline_Ending_Inventory"
}).merge(
    optimized_component[[
        "Week", "Component_ID", "Ending_Inventory"
    ]].rename(columns={
        "Ending_Inventory": "Optimized_Ending_Inventory"
    }),
    on=["Week", "Component_ID"],
    validate="one_to_one"
)

component_buffer_comparison["Inventory_Change"] = (
    component_buffer_comparison["Optimized_Ending_Inventory"]
    - component_buffer_comparison["Baseline_Ending_Inventory"]
)

component_buffer_comparison["Zero_Buffer"] = (
    component_buffer_comparison[
        "Optimized_Ending_Inventory"
    ] == 0
)

# Summarize component-level inventory conditions
component_buffer_summary = (
    component_buffer_comparison.groupby(
        "Component_ID", as_index=False
    )
    .agg(
        Min_Optimized_Inventory=(
            "Optimized_Ending_Inventory", "min"
        ),
        Average_Optimized_Inventory=(
            "Optimized_Ending_Inventory", "mean"
        ),
        Weeks_With_Zero_Buffer=(
            "Zero_Buffer", "sum"
        )
    )
)

# Validate
assert len(component_buffer_comparison) == 36

assert component_buffer_comparison[
    "Optimized_Ending_Inventory"
].ge(0).all()

print("COMPONENT INVENTORY BUFFER ANALYSIS")
print("=" * 45)

display(component_buffer_summary.round(2))

print("\nWEEKS WITH ZERO ENDING INVENTORY:")
display(
    component_buffer_comparison.loc[
        component_buffer_comparison["Zero_Buffer"]
    ][[
        "Week",
        "Component_ID",
        "Optimized_Ending_Inventory"
    ]]
)

print("\nSTEP 5B: PASSED")


COMPONENT INVENTORY BUFFER ANALYSIS


,Component_ID,Min_Optimized_Inventory,Average_Optimized_Inventory,Weeks_With_Zero_Buffer
0,BATT_001,0,60.42,9
1,ELEC_001,0,75.83,8
2,ENCL_001,0,63.33,9



WEEKS WITH ZERO ENDING INVENTORY:


,Week,Component_ID,Optimized_Ending_Inventory
3,2,ELEC_001,0
4,2,BATT_001,0
5,2,ENCL_001,0
6,3,ELEC_001,0
8,3,ENCL_001,0
10,4,BATT_001,0
13,5,BATT_001,0
17,6,ENCL_001,0
18,7,ELEC_001,0
19,7,BATT_001,0



STEP 5B: PASSED


In [8]:
# STEP 6A: Decompose optimization cost improvements

# Baseline costs from Notebook 03
baseline_category_costs = {
    "Procurement":
        data["baseline_costs"]["Procurement_Cost_USD"].sum(),

    "Manufacturing":
        data["baseline_costs"]["Manufacturing_Cost_USD"].sum(),

    "Transportation":
        data["baseline_costs"]["Freight_Cost_USD"].sum(),

    "DC Inventory Holding":
        data["baseline_costs"]["Holding_Cost_USD"].sum()
}

# Optimized costs from Notebook 04
optimized_category_costs = (
    data["optimized_costs"]
    .groupby("Cost_Category")["Cost_USD"]
    .sum()
    .to_dict()
)

assert set(baseline_category_costs) == set(
    optimized_category_costs
)

# Build savings comparison
cost_drivers = pd.DataFrame([
    {
        "Cost_Category": category,
        "Baseline_Cost_USD": baseline_value,
        "Optimized_Cost_USD":
            optimized_category_costs[category]
    }
    for category, baseline_value
    in baseline_category_costs.items()
])

cost_drivers["Savings_USD"] = (
    cost_drivers["Baseline_Cost_USD"]
    - cost_drivers["Optimized_Cost_USD"]
)

# Positive means savings; negative means higher cost
cost_drivers["Contribution_to_Net_Savings_Pct"] = (
    cost_drivers["Savings_USD"] / cost_savings * 100
)

cost_drivers = cost_drivers.sort_values(
    "Savings_USD", ascending=False
).reset_index(drop=True)

# Reconcile savings
assert np.isclose(
    cost_drivers["Baseline_Cost_USD"].sum(),
    baseline_cost
)

assert np.isclose(
    cost_drivers["Optimized_Cost_USD"].sum(),
    optimized_cost
)

assert np.isclose(
    cost_drivers["Savings_USD"].sum(),
    cost_savings
)

print("OPTIMIZATION SAVINGS DECOMPOSITION")
print("=" * 45)

display(cost_drivers.round(2))

print("\nTOTAL SAVINGS: $", round(cost_savings, 2))
print(
    "COST IMPROVEMENT:",
    round(cost_savings / baseline_cost * 100, 2), "%"
)

print("\nSTEP 6A: PASSED")


OPTIMIZATION SAVINGS DECOMPOSITION


,Cost_Category,Baseline_Cost_USD,Optimized_Cost_USD,Savings_USD,Contribution_to_Net_Savings_Pct
0,Transportation,34556.5,25656.5,8900.0,88.92
1,DC Inventory Holding,718.2,119.7,598.5,5.98
2,Procurement,254235.0,253725.0,510.0,5.10
3,Manufacturing,124630.0,124630.0,0.0,0.00



TOTAL SAVINGS: $ 10008.5
COST IMPROVEMENT: 2.42 %

STEP 6A: PASSED


In [9]:
# STEP 6B: Export supplier and manufacturing findings

output_dir = repo_root / "outputs" / "tables"
output_dir.mkdir(parents=True, exist_ok=True)

notebook05_outputs = {
    "supplier_allocation_comparison.csv":
        supplier_comparison,

    "supplier_utilization_comparison.csv":
        supplier_utilization.reset_index(),

    "supplier_weekly_capacity.csv":
        supplier_weekly,

    "supplier_capacity_pressure.csv":
        supplier_pressure,

    "manufacturing_capacity_analysis.csv":
        manufacturing_comparison,

    "component_inventory_buffer_analysis.csv":
        component_buffer_comparison,

    "component_buffer_summary.csv":
        component_buffer_summary,

    "optimization_savings_drivers.csv":
        cost_drivers
}

for filename, dataframe in notebook05_outputs.items():

    file_path = output_dir / filename

    dataframe.to_csv(file_path, index=False)

    assert file_path.is_file()
    assert len(dataframe) > 0

    print("Exported:", filename)

print("\nNOTEBOOK 05 REPORTING EXPORT COMPLETE")
print("Files created:", len(notebook05_outputs))
print("\nSTEP 6B: PASSED")


Exported: supplier_allocation_comparison.csv
Exported: supplier_utilization_comparison.csv
Exported: supplier_weekly_capacity.csv
Exported: supplier_capacity_pressure.csv
Exported: manufacturing_capacity_analysis.csv
Exported: component_inventory_buffer_analysis.csv
Exported: component_buffer_summary.csv
Exported: optimization_savings_drivers.csv

NOTEBOOK 05 REPORTING EXPORT COMPLETE
Files created: 8

STEP 6B: PASSED


In [10]:

# STEP 7: Generate executive summary from actual results

from IPython.display import Markdown, display

# Determine leading savings driver
leading_driver = cost_drivers.iloc[0]

# Manufacturing capacity observations
weeks_at_capacity = int(
    manufacturing_comparison["At_Capacity"].sum()
)

peak_production = int(
    manufacturing_comparison["Optimized_Production"].max()
)

# Supplier capacity observations
supplier_capacity_weeks = int(
    supplier_weekly["At_Capacity"].sum()
)

savings_pct = cost_savings / baseline_cost * 100

summary = f"""
## Final Summary — Supplier & Manufacturing Analysis

### Business Objective
Explain the procurement and manufacturing decisions
behind the optimized supply network.

### Financial Results
- Baseline modeled cost: **${baseline_cost:,.2f}**
- Optimized modeled cost: **${optimized_cost:,.2f}**
- Total modeled savings: **${cost_savings:,.2f}**
- Cost improvement: **{savings_pct:.2f}%**

### Primary Savings Driver
The largest positive cost contribution came from
**{leading_driver['Cost_Category']}**.

- Contribution: **${leading_driver['Savings_USD']:,.2f}**

Individual cost categories may show increases
that offset savings achieved elsewhere.

### Supplier Capacity Findings
- Supplier-component weeks at maximum capacity:
  **{supplier_capacity_weeks}**
- Supplier utilization and allocation differences
  are documented in the exported reporting tables.

### Manufacturing Findings
- Peak optimized weekly production:
  **{peak_production:,} units**
- Weeks operating at plant capacity:
  **{weeks_at_capacity}**
- Weekly production schedules are compared against
  the rule-based baseline.

### Inventory Risk Findings
Component inventory buffers were examined to
identify periods with limited inventory remaining.

Zero ending inventory indicates limited buffer,
not necessarily unmet demand.

### Business Implications
The optimization model demonstrates how coordinated
sourcing, manufacturing timing, transportation,
and inventory planning can improve network cost.

### Limitations
- Historical illustrative demand
- Hypothetical network assumptions
- Simplified hardware bill of materials
- No stochastic demand or supplier variability yet
- Initial inventory acquisition costs excluded
- No real-world company performance claims

### Next Steps
1. Analyze distribution and transportation trade-offs.
2. Test supplier disruptions and demand surges.
3. Evaluate the value of additional capacity.
4. Create executive dashboards and recommendations.
"""

# Render summary within Jupyter
display(Markdown(summary))

# Save standalone Markdown summary
docs_dir = repo_root / "docs"
docs_dir.mkdir(parents=True, exist_ok=True)

summary_file = docs_dir / "notebook05_findings.md"
summary_file.write_text(summary, encoding="utf-8")

assert summary_file.is_file()

print("Executive summary saved:", summary_file)
print("\nNOTEBOOK 05: COMPLETE")



## Final Summary — Supplier & Manufacturing Analysis

### Business Objective
Explain the procurement and manufacturing decisions
behind the optimized supply network.

### Financial Results
- Baseline modeled cost: **$414,139.70**
- Optimized modeled cost: **$404,131.20**
- Total modeled savings: **$10,008.50**
- Cost improvement: **2.42%**

### Primary Savings Driver
The largest positive cost contribution came from
**Transportation**.

- Contribution: **$8,900.00**

Individual cost categories may show increases
that offset savings achieved elsewhere.

### Supplier Capacity Findings
- Supplier-component weeks at maximum capacity:
  **29**
- Supplier utilization and allocation differences
  are documented in the exported reporting tables.

### Manufacturing Findings
- Peak optimized weekly production:
  **580 units**
- Weeks operating at plant capacity:
  **3**
- Weekly production schedules are compared against
  the rule-based baseline.

### Inventory Risk Findings
Component inventory buffers were examined to
identify periods with limited inventory remaining.

Zero ending inventory indicates limited buffer,
not necessarily unmet demand.

### Business Implications
The optimization model demonstrates how coordinated
sourcing, manufacturing timing, transportation,
and inventory planning can improve network cost.

### Limitations
- Historical illustrative demand
- Hypothetical network assumptions
- Simplified hardware bill of materials
- No stochastic demand or supplier variability yet
- Initial inventory acquisition costs excluded
- No real-world company performance claims

### Next Steps
1. Analyze distribution and transportation trade-offs.
2. Test supplier disruptions and demand surges.
3. Evaluate the value of additional capacity.
4. Create executive dashboards and recommendations.


Executive summary saved: /Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-supply-network-optimizer/docs/notebook05_findings.md

NOTEBOOK 05: COMPLETE
